# UK Job Market — Exploratory Data Analysis

**Lesson 36 · Snapshot collected 16 November 2026 · Adzuna UK API**

This is an offline exploratory study of three search-result pages: `data engineer`, `analytics engineer` and `data scientist` (20 returned advertisements per query). It examines record identity, repeated content, provider salary provenance, explicit salary evidence in truncated descriptions, and the coverage of the existing strict salary-verification pattern.

**Research questions**

1. How many records and distinct Adzuna IDs were collected?
2. To what extent does content repeat across advertisements?
3. How frequently is provider salary metadata predicted, and how often do excerpts show a sterling reference?
4. When provider estimates and explicit excerpts can be compared, do they agree?
5. Which real-world salary formats does the current deterministic verification policy support?

**Scope:** These are the first 20 results of each query, not random or representative samples of UK vacancies. Repeated content does not prove repeated vacancies. An excerpt without pay details cannot establish whether the full advertisement discloses pay. No live API or LLM calls occur in this notebook.


## Contents

1. [Data and reproducibility](#data-and-reproducibility)
2. [Data preparation and validation](#data-preparation-and-validation)
3. [Repeated advertisement content](#repeated-advertisement-content)
4. [Salary provenance](#salary-provenance)
5. [Salary references in excerpts](#salary-references-in-excerpts)
6. [Manual comparison of provider estimates](#manual-comparison-of-provider-estimates)
7. [Strict-pattern verification diagnostic](#strict-pattern-verification-diagnostic)
8. [Conclusions and limitations](#conclusions-and-limitations)

Links target explicit HTML anchors in this notebook, rather than relying on automatic heading slug generation. JupyterLab also provides a built-in Table of Contents panel in the left sidebar.


<a id="data-and-reproducibility"></a>

## Data and reproducibility

Run with the repository's Python environment containing pandas, Matplotlib, Pydantic and the installed `uk_job_market_agents` package. Launch JupyterLab from the repository root (or its `notebooks` directory). Put the three saved response JSON files listed in the data-loading cell under `data/raw/`. The metadata sidecars document collection provenance and are optional for execution. Raw responses are excluded from Git.

To reproduce the computations: select **Kernel → Restart Kernel and Run All Cells**. The notebook does not read credentials or write to the input files. All reported values are properties of the collected records, not estimates of the wider labour market.


In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
COLORS = {
    "navy": "#172B4D",
    "teal": "#168A86",
    "amber": "#D49A37",
    "slate": "#677789",
    "light": "#F5F7FA",
}

plt.rcParams.update({
    "figure.figsize": (10, 5),
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelcolor": COLORS["navy"],
    "text.color": COLORS["navy"],
    "grid.alpha": 0.2,
})

<a id="data-preparation-and-validation"></a>

## Data preparation and validation

Load the saved responses with their search-query lineage. We keep repeated-content records in the analytical population rather than dropping them without a vacancy-identity decision. This notebook checks the result count for each snapshot and the presence and uniqueness of IDs in the combined sample. Full ingestion validation was performed separately for the original data-engineering snapshot.


In [ ]:
# Find the repository root from either supported Jupyter launch directory.
cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (candidate for candidate in (cwd, cwd.parent)
     if (candidate / "pyproject.toml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Launch Jupyter from the repository root or notebooks/")

RAW_DATA = PROJECT_ROOT / "data" / "raw"
snapshots = {
    "data_engineering": "adzuna_lesson36_20261116T092204Z.json",
    "analytics_engineering": "adzuna_analytics_engineer_20261116T093217Z.json",
    "data_science": "adzuna_data_scientist_20261116T093219Z.json",
}

records = []
for query_name, filename in snapshots.items():
    with (RAW_DATA / filename).open(encoding="utf-8") as file:
        payload = json.load(file)
    results = payload["results"]
    assert len(results) == 20, f"Unexpected result count in {filename}"
    for job in results:
        records.append({**job, "search_query": query_name})

df = pd.DataFrame(records)
assert len(df) == 60, "Unexpected source snapshot population"
assert df["id"].notna().all(), "Missing Adzuna IDs"
assert df["id"].nunique() == 60, "Source IDs overlap or repeat"
print(f"Observations: {len(df)}")
print(f"Distinct Adzuna IDs: {df['id'].nunique()}")
display(df[["id", "title", "search_query"]].head())


<a id="repeated-advertisement-content"></a>

## Repeated advertisement content

An *exact-content group* is defined by advertiser display name, title and the description excerpt. This is a deliberately narrow identity test: two records can have different Adzuna IDs, locations or salaries but share these fields. Grouping is a **quality diagnostic**, not a determination that the same position was advertised twice.


In [ ]:
# Extract advertiser names from the nested company field.
df["advertiser"] = df["company"].apply(
    lambda company: (
        company.get("display_name")
        if isinstance(company, dict)
        else None
    )
)

# Define an exact-content matching key.
content_columns = [
    "advertiser",
    "title",
    "description",
]

# Identify every record belonging to a repeated-content group.
df["repeated_content"] = df.duplicated(
    subset=content_columns,
    keep=False,
)

# Count distinct content groups and groups with more than one record.
content_group_count = (
    df[content_columns]
    .drop_duplicates()
    .shape[0]
)

print(f"Total advertisements: {len(df)}")
print(f"Distinct content groups: {content_group_count}")
print(f"Records in repeated-content groups: {df['repeated_content'].sum()}")

repeated_group_count = len(
    df.loc[df["repeated_content"], content_columns].drop_duplicates()
)
print(f"Repeated-content groups: {repeated_group_count}")

# Display the affected advertisements.
display(
    df.loc[
        df["repeated_content"],
        ["id", "advertiser", "title", "search_query"],
    ].sort_values(["advertiser", "title"])
)

In [ ]:
# Summarise repeated content by search category.
duplicate_summary = (
    df.groupby("search_query", as_index=False)
    .agg(
        advertisements=("id", "size"),
        repeated_records=("repeated_content", "sum"),
    )
)

duplicate_summary["repeated_rate"] = (
    duplicate_summary["repeated_records"]
    / duplicate_summary["advertisements"]
    * 100
)

labels = {
    "data_engineering": "Data Engineering",
    "analytics_engineering": "Analytics Engineering",
    "data_science": "Data Science",
}

duplicate_summary["role_label"] = (
    duplicate_summary["search_query"].map(labels)
)

display(duplicate_summary)

In [ ]:
# Sort categories by repeated-content rate.
chart_data = duplicate_summary.sort_values(
    "repeated_rate",
    ascending=False,
)

fig, ax = plt.subplots(figsize=(9, 3.5))

bars = ax.barh(
    chart_data["role_label"],
    chart_data["repeated_rate"],
    color=COLORS["teal"],
    height=0.42,
)

# Highest rate at the top.
ax.invert_yaxis()

# Remove axes, ticks and gridlines.
ax.set_xlim(0, 112)
ax.set_xticks([])
ax.tick_params(axis="y", length=0, pad=12)

for spine in ax.spines.values():
    spine.set_visible(False)

ax.grid(False)

# Directly label each bar.
for bar, count, total in zip(
    bars,
    chart_data["repeated_records"],
    chart_data["advertisements"],
):
    percentage = bar.get_width()
    y = bar.get_y() + bar.get_height() / 2

    ax.text(
        percentage + 2,
        y,
        f"{percentage:.0f}%",
        va="center",
        fontsize=12,
        fontweight="bold",
        color=COLORS["navy"],
    )

    ax.text(
        110,
        y,
        f"{int(count)} of {int(total)}",
        ha="right",
        va="center",
        fontsize=10,
        color=COLORS["slate"],
    )

ax.set_title(
    "Repeated advertisement content by search category",
    loc="left",
    fontsize=15,
    fontweight="bold",
    pad=24,
)

fig.text(
    0.125,
    0.01,
    "Share of advertisements matching another record's "
    "advertiser, title and description excerpt.",
    fontsize=9,
    color=COLORS["slate"],
)

plt.tight_layout(rect=(0, 0.08, 1, 1))
plt.show()

### Initial findings

The proportion of advertisements belonging to repeated-content groups varies substantially across the three search samples:

- **Data Engineering:** 10% (2 of 20 advertisements).
- **Analytics Engineering:** 20% (4 of 20).
- **Data Science:** 60% (12 of 20).

The particularly high rate within the Data Science sample is driven by multiple Capital One advertisements sharing identical titles and description excerpts.

These results demonstrate that unique provider identifiers do not necessarily correspond to unique advertisement content.

**Limitation:** The samples consist of the first 20 results returned for each search. They are not random or representative samples of the wider UK labour market. Identical excerpts may also describe genuinely separate vacancies.

Consequently, these percentages describe our collected samples only and should not be interpreted as estimated duplication rates for UK data vacancies.

<a id="salary-provenance"></a>

## Salary provenance

The Adzuna field `salary_is_predicted` describes the provider's prediction status, **not** whether the advertisement itself clearly discloses salary. Provider minimum and maximum fields are not assumed to be verified annual pay; day rates and annualised estimates may occur. Rates here use the 20 records per search as denominator, including potential repeated-content advertisements.


In [ ]:
# Preserve the original provider salary metadata.
df["salary_predicted"] = df["salary_is_predicted"].map({
    "1": True,
    "0": False,
})

# Check for complete provider salary bounds.
df["has_provider_salary"] = (
    df["salary_min"].notna()
    & df["salary_max"].notna()
)

salary_summary = (
    df.groupby("search_query", as_index=False)
    .agg(
        advertisements=("id", "size"),
        predicted=("salary_predicted", lambda s: (s == True).sum()),
        not_predicted=("salary_predicted", lambda s: (s == False).sum()),
        unknown=("salary_predicted", lambda s: s.isna().sum()),
        provider_salary_present=("has_provider_salary", "sum"),
    )
)

salary_summary["role_label"] = (
    salary_summary["search_query"].map(labels)
)

salary_summary["predicted_rate"] = (
    salary_summary["predicted"]
    / salary_summary["advertisements"]
    * 100
)

display(salary_summary)

In [ ]:
chart_data = salary_summary.sort_values(
    "predicted_rate",
    ascending=False,
)

fig, ax = plt.subplots(figsize=(9, 3.5))

y = range(len(chart_data))

not_predicted_pct = (
    chart_data["not_predicted"]
    / chart_data["advertisements"]
    * 100
)

predicted_pct = (
    chart_data["predicted"]
    / chart_data["advertisements"]
    * 100
)

ax.barh(
    y,
    not_predicted_pct,
    color=COLORS["teal"],
    height=0.45,
    label="Not predicted",
)

ax.barh(
    y,
    predicted_pct,
    left=not_predicted_pct,
    color=COLORS["amber"],
    height=0.45,
    label="Predicted",
)

ax.set_yticks(
    list(y),
    chart_data["role_label"],
)

ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xticks([])
ax.tick_params(axis="y", length=0, pad=12)

for spine in ax.spines.values():
    spine.set_visible(False)

ax.grid(False)

for i, (_, row) in enumerate(chart_data.iterrows()):
    n = row["advertisements"]

    left_pct = row["not_predicted"] / n * 100
    right_pct = row["predicted"] / n * 100

    ax.text(
        left_pct / 2,
        i,
        f"{left_pct:.0f}%",
        ha="center",
        va="center",
        color="white",
        fontweight="bold",
    )

    ax.text(
        left_pct + right_pct / 2,
        i,
        f"{right_pct:.0f}%",
        ha="center",
        va="center",
        color=COLORS["navy"],
        fontweight="bold",
    )

ax.set_title(
    "Salary provenance across UK data job searches",
    loc="left",
    pad=25,
)

ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.08),
    ncol=2,
    frameon=False,
)

plt.tight_layout()
plt.show()

**Interpretation.** In this 60-record sample, 42 provider salary records (70%) are marked as predicted and 18 (30%) are not. A non-predicted value is not proof of an explicit annual salary. These percentages should not be generalised to the UK job market, especially given the concentrated repeated content in the Data Science page.


<a id="salary-references-in-excerpts"></a>

## Salary references in excerpts

A `£` character is a **screening signal**, not evidence that a full, valid salary quotation has been extracted. It can occur in annual ranges, daily rates, single-value quotations or ambiguous contexts. Absence of `£` in a truncated excerpt cannot establish non-disclosure in the full advertisement.


In [ ]:
# Identify excerpts containing a pound sterling symbol.
df["has_gbp_reference"] = (
    df["description"]
    .fillna("")
    .str.contains("£", regex=False)
)

# Summarise by search category.
disclosure_summary = (
    df.groupby("search_query", as_index=False)
    .agg(
        advertisements=("id", "size"),
        gbp_references=("has_gbp_reference", "sum"),
    )
)

disclosure_summary["role_label"] = (
    disclosure_summary["search_query"].map(labels)
)

disclosure_summary["reference_rate"] = (
    disclosure_summary["gbp_references"]
    / disclosure_summary["advertisements"]
    * 100
)

display(disclosure_summary)

In [ ]:
salary_candidates = df.loc[
    df["has_gbp_reference"],
    [
        "id",
        "search_query",
        "title",
        "salary_min",
        "salary_max",
        "salary_predicted",
        "description",
    ],
].copy()

print(
    f"Advertisements containing £: "
    f"{len(salary_candidates)} of {len(df)}"
)

display(
    salary_candidates[
        [
            "id",
            "search_query",
            "title",
            "salary_min",
            "salary_max",
            "salary_predicted",
        ]
    ]
)

**Observed:** 12 of the 60 excerpts (20%) contain `£`: eight in the data-engineering sample, four in analytics engineering, and none in data science. Among these 12, ten provider salaries are marked non-predicted and two predicted. This is a *pound-reference rate*, not a verified salary-disclosure rate.


<a id="manual-comparison-of-provider-estimates"></a>

## Manual comparison of provider estimates

We manually reviewed **two selected** advertisements with provider-predicted salaries and visible pound-denominated ranges. The disclosed values below are transcribed from their excerpts. The annual pay period is an **interpretation from the advertised role/context**, rather than an explicit `per year` token in these excerpts; therefore the manually reviewed values are **not automatically production-verified annual salaries**. We compare against the provider's maximum, without claiming that this selected pair measures discrepancy prevalence.


In [ ]:
# Focus on records where both a monetary reference
# and provider-predicted salary metadata exist.

discrepancy_candidates = df.loc[
    df["has_gbp_reference"]
    & df["salary_predicted"].eq(True),
    [
        "id",
        "advertiser",
        "title",
        "salary_min",
        "salary_max",
        "description",
    ],
].copy()

display(
    discrepancy_candidates[
        [
            "id",
            "advertiser",
            "title",
            "salary_min",
            "salary_max",
        ]
    ]
)

In [ ]:
salary_review = pd.DataFrame([
    {
        "id": "5963732722",
        "advertised_min": 75000,
        "advertised_max": 85000,
        "pay_period": "annual (context inferred)",
        "evidence": "Manually reviewed excerpt",
    },
    {
        "id": "5937907052",
        "advertised_min": 55000,
        "advertised_max": 65000,
        "pay_period": "annual (context inferred)",
        "evidence": "Manually reviewed excerpt",
    },
])

comparison = salary_review.merge(
    df[["id", "salary_min", "salary_max"]],
    on="id",
    validate="one_to_one",
)

comparison["provider_exceeds_advertised_max"] = (
    comparison["salary_max"]
    > comparison["advertised_max"]
)

display(comparison)

In [ ]:
comparison["difference_gbp"] = (
    comparison["salary_max"]
    - comparison["advertised_max"]
)

comparison["difference_pct"] = (
    comparison["difference_gbp"]
    / comparison["advertised_max"]
    * 100
)

display(
    comparison[
        [
            "id",
            "advertised_min",
            "advertised_max",
            "salary_max",
            "difference_gbp",
            "difference_pct",
        ]
    ].round(2)
)

### Finding: Provider estimates versus advertised salaries

Adzuna marked the salary values for both reviewed advertisements as predicted. Both excerpts contain a visible sterling range, with an annual period inferred from context.

In both cases, the provider's predicted maximum exceeded the upper bound disclosed in the advertisement excerpt:

- **Lead Data Engineer:** 13.1% above the advertised maximum.
- **Data Engineer:** 25.7% above the advertised maximum.

These observations demonstrate that provider-estimated salaries can materially differ from advertised pay.

**Limitation:** These two records were deliberately selected because they contain both a visible sterling range and provider-predicted values. The findings therefore demonstrate discrepancies, not their prevalence across the wider dataset or UK labour market.

<a id="strict-pattern-verification-diagnostic"></a>

## Strict-pattern verification diagnostic

This stage reuses `SALARY_RANGE_PATTERN`, `parse_salary_range`, `parse_salary_period`, `assess_salary_extraction`, and `is_salary_eligible_for_analytics` from the **existing production package**. To exercise its deterministic contract without OpenAI calls, the notebook scans the pound-bearing excerpts and builds a `SalaryExtractionResult` from the first strict match in each excerpt.

**Interpretation rule:** a `verified` result means that our constructed extraction agrees with the strict parser and supplied evidence. **It is not an independent test of the LLM extractor.** `no_strict_match` is a notebook diagnostic category, not a `SalaryReviewStatus`, and does not mean salary was undisclosed. Annual eligibility is meaningful only for supported and verified *annual* ranges; the notebook does not infer annual pay from amounts alone.


In [ ]:
from uk_job_market_agents.agents.salary_review import (
    SALARY_RANGE_PATTERN,
    assess_salary_extraction,
    parse_salary_range,
    parse_salary_period,
)
from uk_job_market_agents.agents.salary_extractor import (
    SalaryExtractionResult,
)
from uk_job_market_agents.models.salary import SalaryQuote
from uk_job_market_agents.analytics.salary_policy import (
    is_salary_eligible_for_analytics,
)

verification_results = []

for row in df.loc[df["has_gbp_reference"]].itertuples():
    matches = list(
        SALARY_RANGE_PATTERN.finditer(row.description)
    )

    if not matches:
        verification_results.append({
            "id": row.id,
            "title": row.title,
            "supported_pattern_found": False,
            "status": "no_strict_match",
            "annual_eligible": False,
            "evidence": None,
        })
        continue

    evidence = matches[0].group(0)
    bounds = parse_salary_range(evidence)
    period = parse_salary_period(evidence)

    if bounds is None or period is None:
        raise ValueError(
            f"Supported pattern could not be parsed: {row.id}"
        )

    extraction = SalaryExtractionResult(
        salary=SalaryQuote(
            salary_min=bounds[0],
            salary_max=bounds[1],
            period=period,
        ),
        evidence=evidence,
    )

    assessment = assess_salary_extraction(
        row.description,
        extraction,
    )

    verification_results.append({
        "id": row.id,
        "title": row.title,
        "supported_pattern_found": True,
        "status": assessment.status.value,
        "annual_eligible": is_salary_eligible_for_analytics(
            assessment
        ),
        "evidence": evidence,
    })

verification_df = pd.DataFrame(verification_results)

display(verification_df)

print("\nVerification outcomes:")
print(verification_df["status"].value_counts())

print(
    "\nStrict-format annual salary candidates:",
    int(verification_df["annual_eligible"].sum()),
)

### Evidence contexts for pound references

The following read-only diagnostic prints text surrounding the first sterling symbol for each candidate. It is a human-review aid, **not** an alternative salary extractor. It helps distinguish annual-looking ranges, daily rates, approximate salaries and single-sided amounts without silently broadening production verification rules.


In [ ]:
# Display the text surrounding the first pound reference in each excerpt.
# This is an exploratory diagnostic, not salary extraction.

for row in df.loc[df["has_gbp_reference"]].itertuples():
    description = row.description
    position = description.find("£")

    start = max(0, position - 55)
    end = min(len(description), position + 110)

    snippet = description[start:end]

    result = verification_df.loc[
        verification_df["id"] == row.id,
        "status",
    ].iloc[0]

    print(f"\n{row.id} | {row.title}")
    print(f"Verification: {result}")
    print(f"Context: {snippet}")

<a id="conclusions-and-limitations"></a>

## Conclusions and limitations

**Findings from the collected 16 November 2026 snapshots**

- **60 retrieved records / 60 distinct Adzuna IDs** across three search queries; source-ID identity does not establish vacancy identity.
- **47 distinct advertiser/title/excerpt combinations**; 18 observations belong to five repeated-content groups, representing 13 additional records beyond one representative per group. These are potential content repeats, not confirmed duplicate vacancies.
- **42 of 60 (70%)** have provider-predicted salary metadata. Provider predictions are not verified annual disclosures.
- **12 of 60 (20%)** description excerpts contain a pound symbol. This is a screening measure; it cannot establish disclosure prevalence for full advertisements.
- Of these 12, **one** matched the existing strict salary pattern and could be deterministically assessed as verified, but it was a **daily rate**, so **zero** met the strict annual salary eligibility contract in this diagnostic.
- In **two deliberately selected manual comparisons**, the provider-predicted salary exceeded the visible upper bound of the stated range. The manually inferred annual periods should not be confused with strict production verification.

**Important limitations**

The sample comprises first-page search results, not a probability sample; multiple results may come from the same advertiser or copied excerpts. Descriptions are truncated; the record may not describe a currently open vacancy. Company names may identify recruiters rather than employers. The notebook has not run LLM classification/extraction, checked external vacancy pages, validated all 60 records through the full ingestion model, or estimated a population-wide salary distribution. Any external publication must comply with Adzuna's data-redistribution terms.

**Next development decision:** Preserve this notebook as the **baseline**. Design targeted tests for the actual unsupported salary expressions (including decimals, `pa`, missing second currency symbol and one-sided amounts), then determine whether expanding the deterministic verifier is justified. Do not change salary policy purely to increase coverage without protecting evidence provenance and pay-period certainty.
